In [10]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy import stats

import plotly.io as pio
pio.renderers.default = "iframe"

# ── Load data ────────────────────────────────────────────────────────────────
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")

metrics_df = pd.read_csv("data/processed/pft_metrics.csv")

# ── Settings ─────────────────────────────────────────────────────────────────
metrics_to_show = ["correct", "repetitions", "intrusions"]
labels          = ["Correct", "Repetitions\n(inverted)", "Intrusions\n(inverted)"]
INVERT          = {"repetitions", "intrusions"}
groups          = ["HC", "MCI", "Dementia"]
GROUP_COLOURS   = {"HC": "#1D9E75", "MCI": "#BA7517", "Dementia": "#D85A30"}

# Pick one example participant — change this ID to whoever you want
EXAMPLE_ID = metrics_df["participant_id"].iloc[55]
participant_row = metrics_df[metrics_df["participant_id"] == EXAMPLE_ID].iloc[0].to_dict()
diagnosis = participant_row.get("diagnosis", "Unknown")

# ── Per-metric scale ─────────────────────────────────────────────────────────
metric_maxes = {}
for m in metrics_to_show:
    group_max = max(metrics_df.groupby("diagnosis")[m].mean().max(), 1)
    participant_val = participant_row.get(m, 0)
    metric_maxes[m] = max(group_max, participant_val)

def normalise(values):
    normed = []
    for v, m in zip(values, metrics_to_show):
        n = v / metric_maxes[m]
        if m in INVERT:
            n = 1 - n
        normed.append(n)
    return normed

# ── Build chart ──────────────────────────────────────────────────────────────
labels_closed = labels + [labels[0]]
fig = go.Figure()

# Group mean traces
for group in groups:
    group_data = metrics_df[metrics_df["diagnosis"] == group]
    if len(group_data) == 0:
        continue
    raw    = [np.mean(group_data[m].dropna().values) for m in metrics_to_show]
    norm   = normalise(raw)
    closed = norm + [norm[0]]
    colour = GROUP_COLOURS[group]
    r, g, b = int(colour[1:3], 16), int(colour[3:5], 16), int(colour[5:7], 16)

    fig.add_trace(go.Scatterpolar(
        r=closed,
        theta=labels_closed,
        mode="lines",
        name=f"{group} mean",
        line=dict(color=colour, width=2, dash="dash"),
        fill="toself",
        fillcolor=f"rgba({r},{g},{b},0.07)",
        hovertemplate=[
            f"{group}<br>{labels[i]}: {raw[i]:.1f}<extra></extra>"
            for i in range(len(labels))
        ] + [f"{group}<br>{labels[0]}: {raw[0]:.1f}<extra></extra>"],
    ))

# Participant trace
p_raw    = [participant_row.get(m, 0) for m in metrics_to_show]
p_norm   = normalise(p_raw)
p_closed = p_norm + [p_norm[0]]

fig.add_trace(go.Scatterpolar(
    r=p_closed,
    theta=labels_closed,
    mode="lines+markers",
    name=f"Participant {EXAMPLE_ID} ({diagnosis})",
    line=dict(color="#2c3e50", width=3),
    marker=dict(color="#2c3e50", size=9, symbol="diamond"),
    fill="toself",
    fillcolor="rgba(44,62,80,0.1)",
    hovertemplate=[
        f"Participant {EXAMPLE_ID}<br>{labels[i]}: {p_raw[i]}<extra></extra>"
        for i in range(len(labels))
    ] + [f"Participant {EXAMPLE_ID}<br>{labels[0]}: {p_raw[0]}<extra></extra>"],
))

fig.update_layout(
    title=dict(
        text=f"Metric Profile — Participant {EXAMPLE_ID} ({diagnosis})",
        font=dict(size=14)
    ),
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 1],
            tickvals=[0, 0.25, 0.5, 0.75, 1.0],
            ticktext=["0%", "25%", "50%", "75%", "100%"],
            gridcolor="#ddd",
        ),
        angularaxis=dict(gridcolor="#ddd"),
        bgcolor="white",
    ),
    legend=dict(orientation="h", y=-0.15),
    margin=dict(t=60, b=60, l=60, r=60),
    height=500,
    paper_bgcolor="white",
)

fig.write_html("radar_example.html")
print("Open radar_example.html in your browser")


Open radar_example.html in your browser
